## Creating an a summary agent for control performance.
### Approach to be used
- Create an endpoint for a table that will store the control summary
- An agent that will retrive control exceptions and relevant context using RAG.
- An agent should review those exceptions and record an insightful summary.
- An agent should review that summary and determine if it sufficient to be recorded.

Firstly, create and endpoint to be used to store agent feedback

Import all necessary libraries

In [1]:
#import os
from dotenv import load_dotenv
from agents import Agent, Runner,trace, function_tool
import requests
import asyncio
import httpx
from typing import Any
#import json
load_dotenv(override=True)

True

retrieve all the required data

In [ ]:
'/data/exception'

sychronous approach

In [12]:
base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point_list = ['/data/exception','/data/logic','/data/dictionary']  #This takes 27 seconds 
results = {}

for end_point in end_point_list:
    response = requests.get(base_URL+end_point)
    if response.status_code == 200:
        results[end_point.split('/')[-1]] = response.json()
    

In [13]:
print(results)

{'exception': [{'name': 'Charles Hernandez', 'account_number': 'ACC010', 'registration_date': '2019-11-20', 'phone': '+12345678909', 'status': 'Suspended', 'usage_amount': 90.0, 'email': 'charles.hernandez@example.com', 'timestamp': '2023-09-01 08:25:00', 'user_id': 'USR010', 'detection_time': '2026-04-02T16:40:09.087326'}, {'name': 'Isaac Harris', 'account_number': 'ACC042', 'registration_date': '2021-09-12', 'phone': '+12345678941', 'status': 'Suspended', 'usage_amount': 200.0, 'email': 'isaac.harris@example.com', 'timestamp': '2023-09-01 17:50:00', 'user_id': 'USR042', 'detection_time': '2026-04-02T16:40:09.087326'}], 'logic': [{'control_logic': "SELECT *\n                    FROM raw_control_datalake.dev.raw_synthetic_data\n                    WHERE status = 'Suspended';\n                    ", 'created_timestamp': '2026-04-08T10:20:42.380718', 'reference_number': 1, 'control_logic_description': 'Check suspended customers', 'control_logic_status': 'ACTIVE'}], 'dictionary': [{'field

Asychronous Approach

In [14]:


base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point_list = ['/data/exception', '/data/logic', '/data/dictionary'] # This takes 1.2 seconds
async def fetch(client, end_point):
    response = await client.get(base_URL + end_point)
    if response.status_code == 200:
        return end_point.split("/")[-1], response.json()
    return end_point.end_point.split("/")[-1], None

async def fetch_all():
    async with httpx.AsyncClient() as client:
        tasks = [fetch(client, ep) for ep in end_point_list]
        responses = await asyncio.gather(*tasks)
        return {ep: data for ep, data in responses if data is not None}
results = await fetch_all()  

In [15]:
print(results)

{'exception': [{'name': 'Charles Hernandez', 'account_number': 'ACC010', 'registration_date': '2019-11-20', 'phone': '+12345678909', 'status': 'Suspended', 'usage_amount': 90.0, 'email': 'charles.hernandez@example.com', 'timestamp': '2023-09-01 08:25:00', 'user_id': 'USR010', 'detection_time': '2026-04-02T16:40:09.087326'}, {'name': 'Isaac Harris', 'account_number': 'ACC042', 'registration_date': '2021-09-12', 'phone': '+12345678941', 'status': 'Suspended', 'usage_amount': 200.0, 'email': 'isaac.harris@example.com', 'timestamp': '2023-09-01 17:50:00', 'user_id': 'USR042', 'detection_time': '2026-04-02T16:40:09.087326'}], 'logic': [{'control_logic': "SELECT *\n                    FROM raw_control_datalake.dev.raw_synthetic_data\n                    WHERE status = 'Suspended';\n                    ", 'created_timestamp': '2026-04-08T10:20:42.380718', 'reference_number': 1, 'control_logic_description': 'Check suspended customers', 'control_logic_status': 'ACTIVE'}], 'dictionary': [{'field

Create an agent tool 

In [16]:


ALL_ENDPOINTS = ["/data/exception", "/data/logic", "/data/dictionary"]


async def _fetch_one(client: httpx.AsyncClient, endpoint: str) -> tuple[str, Any]:
    BASE_URL = "https://controlweb-supabase.azurewebsites.net"
    key = endpoint.split("/")[-1]
    try:
        response = await client.get(BASE_URL + endpoint)
        if response.status_code == 200:
            return key, response.json()
    except httpx.RequestError:
        pass
    return key, None

@function_tool
async def fetch_controlweb_data(endpoints: list[str] | None = None) -> dict[str, Any]:
    """
    Fetch ControlWeb data from one or more endpoints concurrently.

    Args:
        endpoints: Subset of ['/data/exception', '/data/logic', '/data/dictionary'].
                   Defaults to all three if not provided.

    Returns:
        Dict keyed by endpoint name ('exception', 'logic', 'dictionary').
        Keys for failed/non-200 requests are omitted.
    """
    targets = endpoints if endpoints is not None else ALL_ENDPOINTS
    async with httpx.AsyncClient() as client:
        tasks = [_fetch_one(client, ep) for ep in targets]
        results = await asyncio.gather(*tasks)
    return {key: data for key, data in results if data is not None}

In [17]:
print(fetch_controlweb_data)

FunctionTool(name='fetch_controlweb_data', description='Fetch ControlWeb data from one or more endpoints concurrently.', params_json_schema={'properties': {'endpoints': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'}, {'type': 'null'}], 'description': "Subset of ['/data/exception', '/data/logic', '/data/dictionary'].\n       Defaults to all three if not provided.", 'title': 'Endpoints'}}, 'title': 'fetch_controlweb_data_args', 'type': 'object', 'additionalProperties': False, 'required': ['endpoints']}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7d67185f0800>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)


In [ ]:
system_prompt = f"""You are a Fraud Analyst assistant specializing in the review of automated control exceptions.

## INSTRUCTIONS

You will be given a tool. Before doing any analysis, you MUST call the
`rewrite_data` tool to retrieve the data you need.

Do not proceed with analysis until you get feedback from the rewrite_data tool.

## DATA YOU ARE FETCHING

Each endpoint will return some combination of the following:

- **Data dictionary** — field definitions and value descriptions
- **Control information** — the control's name, description, and exception-generation logic
- **Exception list** — the records flagged by the control

## YOUR TASK

Once all data has been fetched, produce a structured summary covering:

1. **Control Overview**
   What the control is designed to detect and why it matters from a fraud risk perspective —
   in plain language.

2. **Exception Population**
   Volume, key patterns, and notable characteristics of the flagged records.

3. **Analytical Interpretation**
   How the exceptions relate to the control logic. Highlight anything unusual, unexpected,
   or high-priority.

4. **Data Quality Observations**
   Any limitations, gaps, or ambiguities in the data that may affect reliability or
   interpretation.

## GUIDELINES

- Always use the data dictionary to interpret field values accurately.
- Ground all observations strictly in the fetched data — do not speculate.
- Flag ambiguities where control logic or data is unclear.
- Be concise and actionable — prioritise information that helps a reviewer triage or escalate."""

### Creating an agent that will receive the data and convert to an easily data for the processing agent.

In [24]:
agent_1_system_prompt = """

You are an AI assistant that rewrites JSON files into easily processable output for downstream agents.

## INSTRUCTIONS

You will be given a list of endpoints. You MUST call the `fetch_controlweb_data` tool on every endpoint before doing anything else. Do not begin any analysis or rewriting until all fetch calls are complete.

## DATA STRUCTURE

Each endpoint returns some combination of:
- **Data dictionary** — field definitions and value descriptions
- **Control information** — control name, description, and exception-generation logic
- **Exception list** — records flagged by the control

## YOUR TASK

Rewrite the fetched data into a clear, structured format that another agent can easily read and summarise.

"""

In [25]:
Rewrite_data = Agent(name="AI assistant",
                      instructions=agent_1_system_prompt,
                      tools=[fetch_controlweb_data],
                      model="gpt-4o-mini"
                      )

In [19]:
fraud_analyst = Agent(name="Fraud analyst",
                      instructions=system_prompt,
                      tools=[fetch_controlweb_data],
                      model="gpt-4o-mini"
                      )

creating user message for the receipent agent to convert data into readable information

In [26]:
message = f"Here is a list of endpoints :{ALL_ENDPOINTS} use it to extract the information to review"

Checking if the agent works properly

In [27]:
with trace("Extract the required data recent"):
    result = await Runner.run(Rewrite_data,message)

Convert the rewrite agent to a tool

In [29]:
rewrite_tool = Rewrite_data.as_tool(tool_name="rewrite_data", tool_description="an AI assistant that rewrites JSON files into easily processable output for downstream agents")

In [30]:
print(rewrite_tool)

FunctionTool(name='rewrite_data', description='an AI assistant that rewrites JSON files into easily processable output for downstream agents', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7d67187b7710>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)


In [21]:
print(message)

Here is a list of endpoints :['/data/exception', '/data/logic', '/data/dictionary'] use it to extract the information to review


In [22]:
with trace("Extract the required data recent"):
    result = await Runner.run(fraud_analyst,message)

In [23]:
print(result.final_output)

### Control Overview
The control in question is designed to identify accounts that are currently marked as "Suspended." This is significant in the context of fraud risk management because suspended accounts may indicate fraudulent behavior, non-compliance, or risk of financial losses. The control logic uses a query to extract records from a dataset of users, focusing on those with a suspended status, helping to ensure that any fraudulent activities can be flagged and subsequently investigated.

### Exception Population
- **Volume**: There are two exceptions flagged by the control.
- **Key Patterns**:
  - Both accounts are marked as "Suspended."
  - The detected usage amounts are $90.0 and $200.0, which might be considered moderate depending on normal usage patterns.
- **Notable Characteristics**:
  - **Account Names**: Charles Hernandez and Isaac Harris.
  - **Account Numbers**: ACC010 and ACC042.
  - All flagged accounts have a timestamp indicating when they were detected, providing a